# 01 EDA

Day 2 exploratory data analysis for the Lahore AQI forecasting dataset. This notebook reads the cached raw parquet created on Day 1 and focuses on data quality, seasonality, relationships, and temporal dependence that directly inform the locked feature set.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RAW_PATH = Path('../data/raw/aqi_weather_2022-08-01_2026-08-24.parquet')
LOCAL_TIMEZONE = 'Asia/Karachi'

df = pd.read_parquet(RAW_PATH).sort_values('event_time').reset_index(drop=True)
df['local_time'] = df['event_time'].dt.tz_convert(LOCAL_TIMEZONE)
df['hour_local'] = df['local_time'].dt.hour
df['month_name'] = df['local_time'].dt.month_name().str.slice(0, 3)
df['day_name'] = pd.Categorical(
    df['local_time'].dt.day_name(),
    categories=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'],
    ordered=True,
)

aqi_clean = df.dropna(subset=['us_aqi']).copy()
df.head()

,city_id,latitude,longitude,event_time,us_aqi,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,...,relative_humidity_2m,precipitation,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,local_time,hour_local,month_name,day_name
0,lahore,31.5497,74.3436,2022-08-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,96,0.8,975.8,100,9.5,241,2022-08-01 05:00:00+05:00,5,Aug,Monday
1,lahore,31.5497,74.3436,2022-08-01 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,97,2.4,976.4,100,9.4,268,2022-08-01 06:00:00+05:00,6,Aug,Monday
2,lahore,31.5497,74.3436,2022-08-01 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,94,1.5,976.7,95,1.0,225,2022-08-01 07:00:00+05:00,7,Aug,Monday
3,lahore,31.5497,74.3436,2022-08-01 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,91,0.4,976.4,91,2.9,180,2022-08-01 08:00:00+05:00,8,Aug,Monday
4,lahore,31.5497,74.3436,2022-08-01 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,84,0.1,976.7,62,3.6,180,2022-08-01 09:00:00+05:00,9,Aug,Monday


In [2]:
quality_report = pd.DataFrame({
    'missing_pct': (df.isna().mean() * 100).round(3),
    'dtype': df.dtypes.astype(str),
})

event_deltas = df['event_time'].diff().dropna()
gap_summary = event_deltas.value_counts().sort_index()
duplicate_keys = int(df.duplicated(subset=['city_id', 'event_time']).sum())

print('Rows:', len(df))
print('Date range (UTC):', df['event_time'].min(), '->', df['event_time'].max())
print('Duplicate (city_id, event_time):', duplicate_keys)
print('Effective sampling interval counts:')
print(gap_summary.head(10))
display(quality_report.sort_values('missing_pct', ascending=False))

Rows: 35640
Date range (UTC): 2022-08-01 00:00:00+00:00 -> 2026-08-24 23:00:00+00:00
Duplicate (city_id, event_time): 0
Effective sampling interval counts:
event_time
0 days 01:00:00    35639
Name: count, dtype: int64


,missing_pct,dtype
us_aqi,0.269,float64
dust,0.202,float64
pm2_5,0.202,float64
pm10,0.202,float64
carbon_monoxide,0.202,float64
nitrogen_dioxide,0.202,float64
sulphur_dioxide,0.202,float64
ozone,0.202,float64
surface_pressure,0.000,float64
month_name,0.000,str


## Chart 1: Missingness profile

Finding: missingness is very low overall, with AQI and pollutant columns showing only trace nulls near the start of coverage and weather columns effectively complete.

Possible explanation: the CAMS-derived air-quality history starts slightly after the requested range for a few pollutant fields, while ERA5 weather is continuous.

Model implication: we should avoid synthetic filling, accept the short initial null region, and let rolling and lag features naturally drop early rows where context is unavailable.

In [3]:
missing_plot = (df.isna().mean() * 100).sort_values(ascending=False).reset_index()
missing_plot.columns = ['column', 'missing_pct']
fig = px.bar(missing_plot, x='column', y='missing_pct', title='Missing Percentage by Column')
fig.update_layout(xaxis_title='', yaxis_title='Missing %')
fig.show()

## Chart 2: AQI distribution

Finding: the AQI distribution is centered in the unhealthy-for-sensitive-groups to unhealthy range, with a long upper tail and occasional extreme spikes.

Possible explanation: Lahore experiences recurrent pollution episodes rather than rare isolated events, and those episodes can intensify sharply in winter and stagnation periods.

Model implication: we should expect skew and outliers, so median and quantile-aware inspection matters even if the training loss later uses MAE.

In [4]:
fig = px.histogram(aqi_clean, x='us_aqi', nbins=60, title='Distribution of US AQI')
fig.add_vline(x=151, line_dash='dash', line_color='orange')
fig.add_vline(x=201, line_dash='dash', line_color='red')
fig.update_layout(xaxis_title='US AQI', yaxis_title='Count')
fig.show()

## Chart 3: AQI over time

Finding: the time series shows clear multi-month waves, repeated winter peaks, and sustained persistence rather than white-noise behavior.

Possible explanation: seasonal meteorology and emission patterns create long smoothed pollution regimes, which is consistent with the contract note that AQI is already a rolling index.

Model implication: lagged AQI and rolling summaries are essential, and any shuffled validation split would leak these persistent regimes.

In [5]:
daily_aqi = aqi_clean.set_index('event_time')['us_aqi'].resample('D').mean().reset_index()
fig = px.line(daily_aqi, x='event_time', y='us_aqi', title='Daily Mean AQI Over Time')
fig.update_layout(xaxis_title='Date (UTC)', yaxis_title='Daily mean US AQI')
fig.show()

## Chart 4: Monthly seasonality

Finding: monthly mean AQI is highest in winter, especially January, and lowest in spring, with a strong seasonal amplitude.

Possible explanation: winter inversions and seasonal burning behavior can trap pollutants, while spring conditions appear more dispersive.

Model implication: month should be encoded cyclically rather than as a raw integer so the model can learn recurring seasonality without a false December-to-January discontinuity.

In [6]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly = aqi_clean.groupby('month_name', observed=False)['us_aqi'].mean().reindex(month_order).reset_index()
fig = px.bar(monthly, x='month_name', y='us_aqi', title='Average AQI by Month (Local Time)')
fig.update_layout(xaxis_title='Month', yaxis_title='Average US AQI')
fig.show()

## Chart 5: Hour-of-day profile

Finding: AQI is relatively flat overnight and then rises sharply through the late afternoon and early evening local hours.

Possible explanation: daytime activity and boundary-layer changes likely interact to create a recurring intra-day pollution build-up.

Model implication: hour-of-day should be retained as a calendar feature and encoded cyclically, even though the dominant signal is slower-moving seasonality and persistence.

In [7]:
hourly = aqi_clean.groupby('hour_local')['us_aqi'].mean().reset_index()
fig = px.line(hourly, x='hour_local', y='us_aqi', markers=True, title='Average AQI by Local Hour')
fig.update_layout(xaxis_title='Hour (Asia/Karachi)', yaxis_title='Average US AQI')
fig.show()

## Chart 6: Day-of-week profile

Finding: day-of-week differences are present but much smaller than the month and hour effects.

Possible explanation: weekday activity patterns matter somewhat, but AQI in this dataset is dominated more by persistence and seasonal meteorology than by a strong weekly cycle.

Model implication: include day-of-week as a low-cost cyclical feature, but do not expect it to carry the model on its own.

In [8]:
dow = aqi_clean.groupby('day_name', observed=False)['us_aqi'].mean().reset_index()
fig = px.bar(dow, x='day_name', y='us_aqi', title='Average AQI by Day of Week')
fig.update_layout(xaxis_title='', yaxis_title='Average US AQI')
fig.show()

## Chart 7: PM2.5 versus AQI

Finding: PM2.5 has a strong positive relationship with AQI and appears to be one of the most direct physical drivers in the dataset.

Possible explanation: AQI in polluted urban settings is often heavily influenced by particulate concentration, especially during bad-air episodes.

Model implication: PM2.5 lags, rolling means, and change features are justified and should remain in the locked set.

In [9]:
pm25_sample = aqi_clean.sample(n=min(5000, len(aqi_clean)), random_state=42)
fig = px.scatter(pm25_sample, x='pm2_5', y='us_aqi', opacity=0.35, trendline='ols', title='PM2.5 vs AQI')
fig.update_layout(xaxis_title='PM2.5', yaxis_title='US AQI')
fig.show()

## Chart 8: Wind speed versus AQI

Finding: higher wind speeds tend to align with lower AQI, though the relationship is noisy rather than deterministic.

Possible explanation: stronger winds can disperse pollutants, but wind alone cannot fully explain AQI because emissions, chemistry, and seasonality also matter.

Model implication: wind belongs in the feature set, especially as a smoothed 24-hour signal rather than as a single raw point only.

In [10]:
wind_sample = aqi_clean.sample(n=min(5000, len(aqi_clean)), random_state=7)
fig = px.scatter(wind_sample, x='wind_speed_10m', y='us_aqi', opacity=0.35, trendline='ols', title='Wind Speed vs AQI')
fig.update_layout(xaxis_title='Wind speed 10m', yaxis_title='US AQI')
fig.show()

## Chart 9: Humidity and temperature versus AQI

Finding: AQI moves upward with humidity and downward with temperature in this dataset, though both effects are weaker than PM2.5.

Possible explanation: humid, cooler periods may overlap with stagnant pollution conditions, while hotter periods may coincide with stronger vertical mixing.

Model implication: temperature and humidity rolling means are sensible summary features, especially because they can smooth short-lived noise while preserving regime information.

In [11]:
sample = aqi_clean.sample(n=min(5000, len(aqi_clean)), random_state=21)
fig = make_subplots(rows=1, cols=2, subplot_titles=('Temperature vs AQI', 'Humidity vs AQI'))
fig.add_trace(go.Scatter(x=sample['temperature_2m'], y=sample['us_aqi'], mode='markers', marker={'opacity': 0.3}), row=1, col=1)
fig.add_trace(go.Scatter(x=sample['relative_humidity_2m'], y=sample['us_aqi'], mode='markers', marker={'opacity': 0.3}), row=1, col=2)
fig.update_xaxes(title_text='Temperature 2m', row=1, col=1)
fig.update_xaxes(title_text='Relative humidity 2m', row=1, col=2)
fig.update_yaxes(title_text='US AQI', row=1, col=1)
fig.update_yaxes(title_text='US AQI', row=1, col=2)
fig.update_layout(title='Weather Relationships with AQI', showlegend=False)
fig.show()

## Chart 10: Correlation and lag structure

Finding: AQI shows strong positive autocorrelation at 24 hours and meaningful association with particulate variables, while wind is directionally negative.

Possible explanation: the target itself is smoothed and the underlying atmospheric regime persists across days, so yesterday carries real signal into tomorrow.

Model implication: the locked lag and rolling features are justified, and persistence baselines later in the project should be expected to be strong.

In [12]:
corr_cols = ['us_aqi', 'pm2_5', 'pm10', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'surface_pressure']
corr = aqi_clean[corr_cols].corr(numeric_only=True)
lag_df = aqi_clean[['us_aqi']].copy()
lag_df['aqi_lag_24'] = lag_df['us_aqi'].shift(24)
lag_sample = lag_df.dropna().sample(n=min(5000, len(lag_df.dropna())), random_state=99)
autocorr_24 = aqi_clean['us_aqi'].autocorr(lag=24)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Correlation Heatmap', 'AQI vs 24h Lag'))
fig.add_trace(go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, coloraxis='coloraxis'), row=1, col=1)
fig.add_trace(go.Scatter(x=lag_sample['aqi_lag_24'], y=lag_sample['us_aqi'], mode='markers', marker={'opacity': 0.3}), row=1, col=2)
fig.update_xaxes(title_text='AQI lag 24h', row=1, col=2)
fig.update_yaxes(title_text='US AQI', row=1, col=2)
fig.update_layout(title=f'Correlation and Lag Structure (24h autocorr={autocorr_24:.3f})', coloraxis={'colorscale': 'RdBu'})
fig.show()

## Day 2 takeaway

The EDA supports the current contract-driven feature design: strong temporal persistence, strong PM2.5 linkage, meaningful seasonality, mild weekly effects, and weather relationships that are useful but secondary. That is exactly the profile where leakage-safe lags, rolling summaries, cyclical calendar features, and a persistence baseline should all be part of the modelling pipeline.